# CMSC 173 &middot; Module 11 &middot; Example 2
## Iris petals: how many clusters?

First Part 3's six points on a line, where the answer is obvious; then the petal length and width of 150 iris flowers, where the elbow and the silhouette disagree.

**How to use this notebook.** Runtime &rarr; Run all. Every number printed below is on the
slides; change something and see what moves.

**Part 3: six points, a poor start, and choosing K.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

x6 = np.array([0, 1, 10, 11, 20, 21]).reshape(-1, 1)
bad = KMeans(n_clusters=3, init=np.array([[0], [1], [15]]), n_init=1).fit(x6)
good = KMeans(n_clusters=3, n_init=10, random_state=173).fit(x6)
print("poor start:", bad.labels_, bad.inertia_)          # inertia 101.0
print("k-means++, 10 runs:", good.labels_, good.inertia_)  # inertia 1.5
for k in range(1, 6):
    km = KMeans(n_clusters=k, n_init=10, random_state=173).fit(x6)
    sil = silhouette_score(x6, km.labels_) if k > 1 else float("nan")
    print(k, "inertia", round(km.inertia_, 2), "silhouette", round(sil, 3))
# inertia 401.5, 101.5, 1.5, 1.0, 0.5 | silhouette 0.627, 0.898, 0.597, 0.296: both choose K = 3

**Example 2: the iris petals.** Both columns are in centimetres, so no scaling.

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris()
X = iris.data[:, 2:4]                       # petal length, petal width (cm)
km = KMeans(n_clusters=3, n_init=10, random_state=173).fit(X)
print(km.cluster_centers_.round(2))
print(round(km.inertia_, 2), np.bincount(km.labels_))   # 31.37 [50 48 52]

**Elbow and silhouette disagree.**

In [ ]:
ks = range(1, 9)
inertia = [KMeans(n_clusters=k, n_init=10, random_state=173).fit(X).inertia_ for k in ks]
sil = [float(silhouette_score(X, KMeans(n_clusters=k, n_init=10, random_state=173).fit(X).labels_)) for k in ks[1:]]
print("inertia", [round(v, 2) for v in inertia[:6]])     # 550.9, 86.39, 31.37, 19.48, 13.98, 11.04
print("silhouette", [round(v, 3) for v in sil[:5]])      # 0.765, 0.66, 0.613, 0.586, 0.578
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(list(ks), inertia, marker="o"); axes[0].set_title("elbow at K = 3")
axes[1].plot(list(ks[1:]), sil, marker="o", color="green"); axes[1].set_title("silhouette highest at K = 2")
for ax in axes:
    ax.set_xlabel("K")
plt.show()

**Checking against the species**, which K-Means never saw.

In [ ]:
import pandas as pd
from sklearn.metrics import adjusted_rand_score

print(pd.crosstab(iris.target_names[iris.target], km.labels_))
print("agree:", pd.crosstab(iris.target, km.labels_).max(axis=1).sum(), "of 150")   # 144
print("ARI", round(adjusted_rand_score(iris.target, km.labels_), 3))               # 0.886

**Try it.** Use all four measurements, standardised (`StandardScaler`). Do the elbow and the silhouette still disagree? Does the ARI go up?